# Something's Missing Here — the Keras siamese flow

**Object DeTaken**, by Shay Gil and Michal Peri Markov.

Two photos of the same scene in, one boolean out: *did anything go
missing?* This notebook is the complete, self-contained Keras 3 +
EfficientNetB0 flow — the implementation that gave the project its best
results:

1. install dependencies and download the
   [`Shay85Gil/somethings_missing_here`](https://huggingface.co/datasets/Shay85Gil/somethings_missing_here)
   dataset (~2.8 GB),
2. read the before/after pairs and hold out whole scenes for validation,
3. train a siamese EfficientNetB0 ("did anything go missing?") with a
   frozen-then-unfrozen backbone schedule,
4. pick the decision threshold on validation (Youden's J),
5. evaluate on the held-out test scenes and look at the failures.

Runs top to bottom as a single file — CPU works but a GPU (e.g. a free
Colab T4) makes it pleasant. Keras uses the torch backend so the GPU is
used wherever torch finds one.

In [ ]:
%pip install -q "keras>=3" torch torchvision "albumentations<2" \
    opencv-python-headless scikit-learn huggingface_hub matplotlib

In [ ]:
import json
import os
import random
import time
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "torch")       # GPU via torch
os.environ.setdefault("NO_ALBUMENTATIONS_UPDATE", "1")

import albumentations as A
import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import layers
from sklearn.metrics import roc_auc_score, roc_curve

CFG = dict(
    size=(256, 256),     # model input (height, width)
    bs=16,               # batch size
    epochs=30,           # upper bound; early stopping usually ends sooner
    freeze_epochs=10,    # backbone frozen for the first N epochs
    patience=10,         # stop after N epochs without validation improvement
    dropout=0.4,
    hidden=128,          # width of the head's hidden layer
    activation="relu",
    val_fraction=0.2,    # fraction of rows (whole scenes) held out
    seed=42,
    cache_side=640,      # decode cache: images resized to this max side
)
keras.utils.set_random_seed(CFG["seed"])
BLUE, ORANGE = "#2a78d6", "#eb6834"
print("keras", keras.version(), "| backend:", keras.backend.backend())

## 1. Dataset

559 viewpoint-aligned before/after pairs with labels naming the removed
items — original photographs plus pairs derived from Remove360. The
download is skipped if the folder already exists.

In [ ]:
DATA = Path("somethings_missing_here")
if not (DATA / "training").is_dir():
    from huggingface_hub import snapshot_download
    snapshot_download("Shay85Gil/somethings_missing_here",
                      repo_type="dataset", local_dir=DATA)
TRAIN_ROOT, TEST_ROOT = DATA / "training", DATA / "test"
print("training pairs folder:", TRAIN_ROOT, "| test:", TEST_ROOT)

In [ ]:
def load_pairs(root):
    """Pair folders {before.jpg, after.jpg, label.json} at any depth;
    the parent path relative to the root is the pair's scene id."""
    root, rows = Path(root), []
    for label_file in sorted(root.rglob("label.json")):
        pair_dir = label_file.parent
        before, after = pair_dir / "before.jpg", pair_dir / "after.jpg"
        if not (before.is_file() and after.is_file()):
            continue
        meta = json.loads(label_file.read_text())
        rows.append(dict(
            before=str(before), after=str(after),
            y=1.0 if meta.get("missing") else 0.0,
            scene=str(pair_dir.parent.relative_to(root)),
            items=meta.get("items", []), pair=pair_dir.name))
    return rows


def scene_split(rows, seed, val_fraction):
    """(train_rows, val_rows) with WHOLE scenes held out -- the same
    physical location never appears on both sides, which is what keeps
    validation honest."""
    scenes = sorted({r["scene"] for r in rows})
    counts = {s: sum(r["scene"] == s for r in rows) for s in scenes}
    order = list(np.random.default_rng(seed).permutation(scenes))
    target = max(1, round(val_fraction * len(rows)))
    held, held_count = set(), 0
    for scene in order[:-1]:              # always leave >= 1 scene training
        held.add(scene)
        held_count += counts[scene]
        if held_count >= target:
            break
    return ([r for r in rows if r["scene"] not in held],
            [r for r in rows if r["scene"] in held])


rows = load_pairs(TRAIN_ROOT)
tr_rows, va_rows = scene_split(rows, CFG["seed"], CFG["val_fraction"])
print(f"{len(rows)} pairs, {len({r['scene'] for r in rows})} scenes | "
      f"train {len(tr_rows)} / val {len(va_rows)} "
      f"({len({r['scene'] for r in va_rows})} held-out scenes)")
print(f"positives: {int(sum(r['y'] for r in rows))} | "
      f"negatives: {int(sum(1 - r['y'] for r in rows))}")

### A look at the data

Left-to-right: `before` on top, `after` below. A **positive** pair has
something removed between the two shots; a **negative** pair shows the
same scene twice with nothing missing (only viewpoint/lighting change).

In [ ]:
def imread_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show_pairs(sample, title):
    fig, axes = plt.subplots(2, len(sample), figsize=(3.2 * len(sample), 5.6))
    for col, r in enumerate(sample):
        for row, key in enumerate(("before", "after")):
            ax = axes[row, col]
            ax.imshow(imread_rgb(r[key]))
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0:
                gone = ", ".join(r["items"]) if r["items"] else "nothing"
                ax.set_title(f"{r['scene']}\nmissing: {gone}", fontsize=9)
    axes[0, 0].set_ylabel("before", fontsize=11)
    axes[1, 0].set_ylabel("after", fontsize=11)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


rng = random.Random(CFG["seed"])
show_pairs(rng.sample([r for r in rows if r["y"] == 1], 4),
           "positive pairs -- something was removed")
show_pairs(rng.sample([r for r in rows if r["y"] == 0], 4),
           "negative pairs -- nothing removed")

In [ ]:
# label balance per collection (top-level folder of each scene)
collections = sorted({r["scene"].split("/")[0] for r in rows})
pos = [sum(r["y"] for r in rows if r["scene"].startswith(c)) for c in collections]
neg = [sum(1 - r["y"] for r in rows if r["scene"].startswith(c)) for c in collections]

x = np.arange(len(collections))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(x - 0.2, pos, 0.38, color=BLUE, label="positive (missing)")
ax.bar(x + 0.2, neg, 0.38, color=ORANGE, label="negative (no change)")
ax.set_xticks(x, collections)
ax.set_ylabel("pairs")
ax.set_title("training pairs by collection", fontsize=11)
ax.grid(axis="y", color="#000000", alpha=0.08)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.show()

## 2. Augmentation and batching

The augmentation design carries the core insight of the whole task:

- **geometric transforms are shared** by both images (one random affine
  applied identically) — a camera move is not a removal;
- **photometric transforms are independent** per image — this is what
  teaches "the light changed" ≠ "the item is gone";
- the `after` image gets a small extra **jitter** so the model tolerates
  imperfect registration.

Images stay 0–255 float: EfficientNet embeds its own input rescaling.

In [ ]:
def build_transforms(size, train):
    h, w = size
    shared = A.Compose([
        A.LongestMaxSize(max_size=max(h, w)),
        A.PadIfNeeded(h, w, border_mode=cv2.BORDER_CONSTANT),
        A.CenterCrop(h, w),
    ] + ([A.Affine(scale=(0.9, 1.1), translate_percent=0.05,
                   rotate=(-7, 7), p=0.7)] if train else []),
        additional_targets={"image2": "image"})
    jitter = (A.Affine(translate_percent=0.02, rotate=(-2, 2), p=0.5)
              if train else None)
    photo = A.Compose([
        A.RandomBrightnessContrast(0.3, 0.3, p=0.8),
        A.HueSaturationValue(12, 20, 12, p=0.5),
        A.GaussNoise(p=0.2),
    ]) if train else None
    return shared, jitter, photo


class ImageCache:
    """Decode each JPEG once, keep it in RAM (max side CFG['cache_side'])."""

    def __init__(self, max_side):
        self.max_side, self._store = max_side, {}

    def get(self, path):
        img = self._store.get(path)
        if img is None:
            img = imread_rgb(path)
            h, w = img.shape[:2]
            s = self.max_side / max(h, w)
            if s < 1.0:
                img = cv2.resize(img, (round(w * s), round(h * s)),
                                 interpolation=cv2.INTER_AREA)
            self._store[path] = img
        return img


class PairSequence(keras.utils.PyDataset):
    """Yields ((before, after), y) float32 batches with the augmentation
    contract above."""

    def __init__(self, rows, size, train, cache, batch_size, seed=0, **kw):
        super().__init__(**kw)
        self.rows, self.cache, self.batch_size = list(rows), cache, batch_size
        self.shuffle = train
        self.rng = np.random.default_rng(seed)
        self.order = np.arange(len(self.rows))
        self.shared, self.jitter, self.photo = build_transforms(size, train)
        if self.shuffle:
            self.rng.shuffle(self.order)

    def _sample(self, r):
        out = self.shared(image=self.cache.get(r["before"]),
                          image2=self.cache.get(r["after"]))
        a, b = out["image"], out["image2"]
        if self.jitter is not None:
            b = self.jitter(image=b)["image"]
        if self.photo is not None:
            a = self.photo(image=a)["image"]
            b = self.photo(image=b)["image"]
        return a.astype(np.float32), b.astype(np.float32), np.float32(r["y"])

    def __len__(self):
        return (len(self.rows) + self.batch_size - 1) // self.batch_size

    def __getitem__(self, i):
        idx = self.order[i * self.batch_size:(i + 1) * self.batch_size]
        s = [self._sample(self.rows[j]) for j in idx]
        return ((np.stack([x[0] for x in s]), np.stack([x[1] for x in s])),
                np.array([x[2] for x in s], dtype=np.float32))

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.order)


cache = ImageCache(CFG["cache_side"])
t0 = time.time()
for r in rows:
    cache.get(r["before"]); cache.get(r["after"])
print(f"cached {len(cache._store)} images in {time.time() - t0:.0f}s")

tr_seq = PairSequence(tr_rows, CFG["size"], True, cache, CFG["bs"],
                      seed=CFG["seed"])
va_seq = PairSequence(va_rows, CFG["size"], False, cache, CFG["bs"])

In [ ]:
# the same pair, augmented three times: geometry moves together,
# lighting/colour drift apart
preview = PairSequence([tr_rows[0]], CFG["size"], True, cache, 1,
                       seed=CFG["seed"])
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.8))
for col in range(3):
    a, b, _ = preview._sample(tr_rows[0])
    axes[0, col].imshow(a.astype(np.uint8))
    axes[1, col].imshow(b.astype(np.uint8))
    for row in range(2):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
axes[0, 0].set_ylabel("before", fontsize=11)
axes[1, 0].set_ylabel("after", fontsize=11)
fig.suptitle("one pair, three augmentation draws", fontsize=12)
fig.tight_layout()
plt.show()

## 3. The model

**One** EfficientNetB0 applied to both inputs — calling the same layer
instance twice shares the weights; that sharing *is* the siamese part.
The head reads `[f_before, f_after, f_before − f_after]`: the **signed**
difference lets the model tell removal from addition (`abs()` could
not). LayerNorm instead of BatchNorm because batches are small.

In [ ]:
def build_model(size, p_drop, hidden, activation):
    h, w = size
    backbone = keras.applications.EfficientNetB0(
        include_top=False, pooling="avg", weights="imagenet",
        input_shape=(h, w, 3))
    inp_a = keras.Input((h, w, 3), name="before")
    inp_b = keras.Input((h, w, 3), name="after")
    fa, fb = backbone(inp_a), backbone(inp_b)
    x = layers.Concatenate()([fa, fb, layers.Subtract()([fa, fb])])
    x = layers.Dropout(p_drop)(x)
    x = layers.Dense(hidden)(x)
    x = layers.LayerNormalization()(x)
    x = layers.Activation(activation)(x)
    x = layers.Dropout(p_drop)(x)
    out = layers.Dense(1, name="logit")(x)
    model = keras.Model([inp_a, inp_b], out)
    model.backbone = backbone
    return model


def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
        loss=keras.losses.BinaryCrossentropy(from_logits=True))


model = build_model(CFG["size"], CFG["dropout"], CFG["hidden"],
                    CFG["activation"])
model.backbone.trainable = False     # frozen BN stays in inference mode
compile_model(model, lr=1e-3)
print(f"parameters: {model.count_params():,} "
      f"(backbone counted once -- shared weights)")

## 4. Training

Backbone frozen for the first `freeze_epochs` (only the head learns),
then everything fine-tunes at a low learning rate. After each epoch the
validation set picks both the **best checkpoint** and the **decision
threshold** — Youden's J, the ROC point maximising TPR − FPR — because a
BCE-trained net on imbalanced data has no reason to be calibrated
around 0.5.

In [ ]:
def predict_probs(model, seq):
    ps, ys = [], []
    for i in range(len(seq)):
        (a, b), y = seq[i]
        logits = model.predict_on_batch((a, b)).squeeze(-1)
        ps.append(1.0 / (1.0 + np.exp(-logits)))
        ys.append(y)
    return np.concatenate(ps), np.concatenate(ys)


def best_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(np.clip(thr[np.argmax(tpr - fpr)], 0.01, 0.99))


best = {"auc": -1.0, "thr": 0.5, "weights": None, "epoch": -1}
history, patience = [], 0

for epoch in range(CFG["epochs"]):
    if epoch == CFG["freeze_epochs"]:
        model.backbone.trainable = True
        compile_model(model, lr=1e-4)

    model.fit(tr_seq, epochs=1, verbose=0)
    p, y = predict_probs(model, va_seq)
    auc = roc_auc_score(y, p)
    thr = best_threshold(y, p)
    acc = float(((p > thr) == (y == 1)).mean())
    history.append({"auc": auc, "acc": acc})

    if auc > best["auc"]:
        best = {"auc": auc, "thr": thr, "weights": model.get_weights(),
                "epoch": epoch}
        patience = 0
    else:
        patience += 1
    print(f"epoch {epoch + 1:2d}/{CFG['epochs']} | val AUC {auc:.3f} "
          f"acc {acc:.3f} | best {best['auc']:.3f} @ {best['epoch'] + 1} "
          f"| patience {patience}/{CFG['patience']}")
    if patience >= CFG["patience"]:
        print("early stop")
        break

model.set_weights(best["weights"])
THRESHOLD = best["thr"]
print(f"\nkept epoch {best['epoch'] + 1}: val AUC {best['auc']:.3f}, "
      f"threshold {THRESHOLD:.3f}")

In [ ]:
# training curves: the dot marks the epoch whose weights were kept
fig, ax = plt.subplots(figsize=(7, 4.2))
for key, label, color in (("auc", "val AUC", BLUE),
                          ("acc", "val accuracy", ORANGE)):
    curve = [h[key] for h in history]
    ax.plot(range(1, len(curve) + 1), curve, color=color, linewidth=2,
            label=label)
    ax.plot(best["epoch"] + 1, curve[best["epoch"]], "o", color=color,
            markersize=6)
ax.axvline(CFG["freeze_epochs"] + 0.5, color="#8a8a86", linewidth=1,
           linestyle="--")
ax.text(CFG["freeze_epochs"] + 0.7, ax.get_ylim()[0] + 0.01,
        "backbone unfrozen", fontsize=8, color="#8a8a86")
ax.set_xlabel("epoch")
ax.grid(axis="y", color="#000000", alpha=0.08)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.show()

## 5. Held-out test evaluation

The test scenes were never seen in training. Only natural pairs — no
synthetic imagery, no reversals.

In [ ]:
test_rows = load_pairs(TEST_ROOT)
for r in test_rows:
    cache.get(r["before"]); cache.get(r["after"])
test_seq = PairSequence(test_rows, CFG["size"], False, cache, CFG["bs"])

p, y = predict_probs(model, test_seq)
pred = p > THRESHOLD
tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
auc = roc_auc_score(y, p)

print(f"{len(test_rows)} test pairs | threshold {THRESHOLD:.3f}")
print(f"AUC {auc:.3f}   acc {(tp + tn) / len(y):.3f}")
print(f"TP {tp}  FP {fp}  FN {fn}  TN {tn}")
if tp + fp: print(f"precision {tp / (tp + fp):.3f}", end="   ")
if tp + fn: print(f"recall {tp / (tp + fn):.3f}")

In [ ]:
# ROC curve and where the scores land relative to the threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4))

fpr, tpr, _ = roc_curve(y, p)
ax1.plot(fpr, tpr, color=BLUE, linewidth=2)
ax1.plot([0, 1], [0, 1], color="#8a8a86", linewidth=1, linestyle="--")
ax1.set_xlabel("false positive rate"); ax1.set_ylabel("true positive rate")
ax1.set_title(f"test ROC (AUC {auc:.3f})", fontsize=11)

bins = np.linspace(0, 1, 21)
ax2.hist(p[y == 1], bins=bins, color=BLUE, alpha=0.75,
         label="true: missing")
ax2.hist(p[y == 0], bins=bins, color=ORANGE, alpha=0.75,
         label="true: no change")
ax2.axvline(THRESHOLD, color="#333333", linewidth=1.5, linestyle="--")
ax2.text(THRESHOLD + 0.01, ax2.get_ylim()[1] * 0.92, "threshold",
         fontsize=8)
ax2.set_xlabel("predicted probability"); ax2.set_ylabel("pairs")
ax2.set_title("score distribution", fontsize=11)
ax2.legend(frameon=False, fontsize=9)
for ax in (ax1, ax2):
    ax.grid(color="#000000", alpha=0.08)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

### The failures

The most confidently wrong pairs — misses (`missing` but the model said
no) and false alarms (nothing missing but the model fired).

In [ ]:
failures = [(i, abs(p[i] - y[i])) for i in range(len(test_rows))
            if (p[i] > THRESHOLD) != (y[i] == 1)]
failures.sort(key=lambda t: -t[1])
print(f"{len(failures)} failures / {len(test_rows)} pairs")

show = failures[:4]
if show:
    fig, axes = plt.subplots(2, len(show),
                             figsize=(3.2 * len(show), 5.8), squeeze=False)
    for col, (i, _) in enumerate(show):
        r = test_rows[i]
        kind = "MISSED" if y[i] == 1 else "FALSE ALARM"
        items = ", ".join(r["items"]) or "-"
        for row, key in enumerate(("before", "after")):
            ax = axes[row, col]
            ax.imshow(imread_rgb(r[key]))
            ax.set_xticks([]); ax.set_yticks([])
        axes[0, col].set_title(
            f"{kind}  p={p[i]:.2f}\n{r['scene']}/{r['pair']}\n"
            f"items: {items}", fontsize=8)
    axes[0, 0].set_ylabel("before", fontsize=11)
    axes[1, 0].set_ylabel("after", fontsize=11)
    fig.tight_layout()
    plt.show()

## 6. Save

The checkpoint plus a JSON sidecar carrying the decision threshold —
everything needed to serve predictions
(`prob = sigmoid(model([before, after])) > threshold`).

In [ ]:
model.save("kmodel.keras")
Path("kmodel.json").write_text(json.dumps({
    "size": list(CFG["size"]), "threshold": THRESHOLD,
    "val_auc": best["auc"], "backbone": "efficientnetb0",
    "dropout": CFG["dropout"], "hidden": CFG["hidden"],
    "activation": CFG["activation"],
}))
print("saved kmodel.keras + kmodel.json "
      f"(threshold {THRESHOLD:.3f})")